In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\bunyo\OneDrive\Desktop\AI_Course\ModularProgramProjects\finalProject\data\sampled_data\sampled_data.csv")

In [3]:
df.head()

,Efficiency,Weight_kg,Acceleration(0-100),Firth_stop_range_km,Battery_kWh,Fast_charger(kW),Towing_kg,Cargo_volume,Price/Range(km),FastCharge_time_hrs,Energy_weight_ratio,Added_Range_1Stop,Range_km_level,Price_level
0,0.473118,0.577071,0.343137,0.418936,0.539171,0.272727,0.400000,0.2750,0.050938,0.443576,0.583782,0.366935,1,1
1,0.408602,0.612235,0.171569,0.656291,0.649770,0.630303,0.666667,0.3335,0.079088,0.223483,0.697531,0.741935,2,1
2,0.344086,0.677264,0.200980,0.690013,0.751152,0.427273,0.250000,0.2150,0.067024,0.372340,0.764383,0.463710,2,2
3,0.543011,0.479769,0.372549,0.351492,0.612903,0.151515,0.000000,0.2335,0.049598,0.885938,0.781900,0.096774,1,1
4,0.392473,0.557322,0.058824,0.619974,0.603687,0.593939,0.000000,0.1855,0.120643,0.223214,0.688162,0.709677,1,2


In [4]:
df = df.dropna()
x = df.drop(columns=['Range_km_level','Price_level'])
y = df[['Range_km_level','Price_level']]

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier

base_model = XGBClassifier(random_state=42)
model = MultiOutputClassifier(base_model)


param_grid = {
    'estimator__n_estimators': [50, 100, 200],   
    'estimator__max_depth': [3, 5, 7],            
    'estimator__learning_rate': [0.01, 0.1, 0.2]  
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,                 
    n_jobs=-1,          
    verbose=2           
)

grid_search.fit(x, y)

print("Eng yaxshi parametrlar:", grid_search.best_params_)
print("Eng yuqori aniqlik (Accuracy):", grid_search.best_score_)


best_model = grid_search.best_estimator_

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Eng yaxshi parametrlar: {'estimator__learning_rate': 0.2, 'estimator__max_depth': 3, 'estimator__n_estimators': 200}
Eng yuqori aniqlik (Accuracy): 0.900839054157132


In [ ]:
x = df.drop(columns=['Range_km_level','Price_level'])
y = df['Range_km_level']

import optuna
import numpy as np
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score


def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True), 
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
        
    }
    
    model = XGBClassifier(**param)
    
    try:
        scores = cross_val_score(model, x, y, cv=3, scoring='accuracy', n_jobs=-1, error_score='raise')
    except Exception as e:
        print(f"Xatolik yuz berdi: {e}")
        return 0.0 
    
    return np.mean(scores)
    

study = optuna.create_study(direction='maximize', study_name="EV_MultiOutput_Tuning")


study.optimize(objective, n_trials=20)


print("\nEng yaxshi parametrlar:", study.best_params)
print("Eng yuqori aniqlik (Accuracy):", study.best_value)

[I 2026-03-15 12:50:17,307] A new study created in memory with name: EV_MultiOutput_Tuning


Optuna qidiruvni boshladi...


[I 2026-03-15 12:50:21,333] Trial 0 finished with value: 0.9710144927536232 and parameters: {'n_estimators': 209, 'max_depth': 7, 'learning_rate': 0.09649198846483238, 'subsample': 0.7673929000310953, 'colsample_bytree': 0.6612807388599283}. Best is trial 0 with value: 0.9710144927536232.
[I 2026-03-15 12:50:24,199] Trial 1 finished with value: 0.9588100686498855 and parameters: {'n_estimators': 146, 'max_depth': 5, 'learning_rate': 0.019226889326134335, 'subsample': 0.9075580281956598, 'colsample_bytree': 0.8201426117180273}. Best is trial 0 with value: 0.9710144927536232.
[I 2026-03-15 12:50:27,263] Trial 2 finished with value: 0.9725400457665904 and parameters: {'n_estimators': 284, 'max_depth': 7, 'learning_rate': 0.04425810633030291, 'subsample': 0.6500329747503877, 'colsample_bytree': 0.6097020370524835}. Best is trial 2 with value: 0.9725400457665904.
[I 2026-03-15 12:50:27,791] Trial 3 finished with value: 0.969488939740656 and parameters: {'n_estimators': 169, 'max_depth': 8, 


Eng yaxshi parametrlar: {'n_estimators': 284, 'max_depth': 7, 'learning_rate': 0.04425810633030291, 'subsample': 0.6500329747503877, 'colsample_bytree': 0.6097020370524835}
Eng yuqori aniqlik (Accuracy): 0.9725400457665904


In [13]:
x = df.drop(columns=['Range_km_level','Price_level'])
y = df['Price_level']

import optuna
import numpy as np
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score


def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True), 
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
        
    }
    
    model = XGBClassifier(**param)
    
    try:
        scores = cross_val_score(model, x, y, cv=3, scoring='accuracy', n_jobs=-1, error_score='raise')
    except Exception as e:
        print(f"Xatolik yuz berdi: {e}")
        return 0.0 
    
    return np.mean(scores)
    

study = optuna.create_study(direction='maximize', study_name="EV_MultiOutput_Tuning")


study.optimize(objective, n_trials=20)


print("\nEng yaxshi parametrlar:", study.best_params)
print("Eng yuqori aniqlik (Accuracy):", study.best_value)

[I 2026-03-15 12:58:43,456] A new study created in memory with name: EV_MultiOutput_Tuning
[I 2026-03-15 12:58:47,490] Trial 0 finished with value: 0.9389778794813118 and parameters: {'n_estimators': 262, 'max_depth': 9, 'learning_rate': 0.22525397510067854, 'subsample': 0.734184396022083, 'colsample_bytree': 0.8596482399438148}. Best is trial 0 with value: 0.9389778794813118.
[I 2026-03-15 12:58:50,477] Trial 1 finished with value: 0.9359267734553777 and parameters: {'n_estimators': 274, 'max_depth': 8, 'learning_rate': 0.23334124366278816, 'subsample': 0.8508895155813012, 'colsample_bytree': 0.6051873707745267}. Best is trial 0 with value: 0.9389778794813118.
[I 2026-03-15 12:58:54,441] Trial 2 finished with value: 0.9252479023646072 and parameters: {'n_estimators': 230, 'max_depth': 9, 'learning_rate': 0.011633624883567591, 'subsample': 0.7952207660496218, 'colsample_bytree': 0.6117870574002058}. Best is trial 0 with value: 0.9389778794813118.
[I 2026-03-15 12:58:54,915] Trial 3 fin


Eng yaxshi parametrlar: {'n_estimators': 177, 'max_depth': 10, 'learning_rate': 0.29453710821009343, 'subsample': 0.7599428695271804, 'colsample_bytree': 0.8251782093968107}
Eng yuqori aniqlik (Accuracy): 0.94279176201373
